In [144]:
import warnings
warnings.filterwarnings('ignore')

import os
import pickle

import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

from pandas.tseries.offsets import MonthEnd, MonthBegin
from maricovault.MaricoDB import MaricoSnowflake

from joblib import Parallel, delayed

In [145]:
base_dir = '/data/aman_singh/acuuracy_check'
os.chdir(base_dir)

In [146]:
def get_dbconnection(db_name): 

    KEY_VAULT_NAME = "prod-pwd"
    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection

In [147]:
dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


### Helper functions

In [148]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

realignment_df = realignment_df[
    realignment_df['channel'].isin(['QCOM', 'QCOM B2C', 'ALL'])]


def realign_pskus(data, column):
    realignment_data = realignment_df.copy()
    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']

    
    data[column] = data[column].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data[column] == old_psku, column
        ] = new_psku

    return data

In [149]:
blinkit_forecast_apr = pd.read_csv('/data/aman_singh/acuuracy_check/Marico Ltd._forecast_Apr 2026_to_Jul 2026.csv')

blinkit_forecast_apr.rename(columns = {'item_id':'item_code', 'Apr - forecast':'forecast_quantity'}, inplace = True)
blinkit_forecast_apr.drop(columns = ['May - forecast', 'Jun - forecast', 'Jul - forecast'],inplace = True)
blinkit_forecast_apr['date'] = '2026-04-30'
blinkit_forecast_apr['date'] = pd.to_datetime(blinkit_forecast_apr['date'])
blinkit_forecast_apr

,facility_id,facility_name,item_code,item_name,manufacturer_id,manufacturer_name,vendor_id,vendor_name,forecast_quantity,date
0,92,Super Store Dasna 2 - Warehouse,10224608,Parachute Advansed Gold Vitamin E Coconut Hair...,1385,Marico Ltd.,5271.0,Marico Ltd,134,2026-04-30
1,92,Super Store Dasna 2 - Warehouse,10010405,Parachute Advansed Aloe Vera Hair Oil for Soft...,1385,Marico Ltd.,5271.0,Marico Ltd,192,2026-04-30
2,92,Super Store Dasna 2 - Warehouse,10221231,Just Herbs 12 Chemical Free Nail Paint (Burgun...,1385,Marico Ltd.,5271.0,Marico Ltd,0,2026-04-30
3,92,Super Store Dasna 2 - Warehouse,10064182,Saffola Honey Active (500 g)(Pack)500 gm - Rs ...,1385,Marico Ltd.,5271.0,Marico Ltd,20,2026-04-30
4,92,Super Store Dasna 2 - Warehouse,10211385,Livon Nourishing Hair Serum(Bottle)100 ml - Rs...,1385,Marico Ltd.,5271.0,Marico Ltd,1,2026-04-30
...,...,...,...,...,...,...,...,...,...,...
5040,2468,Nagpur N1 - Feeder Warehouse,10181271,Saffola Classic Masala Flavoured Oats750 g - R...,1385,Marico Ltd.,5271.0,Marico Ltd,220,2026-04-30
5041,2468,Nagpur N1 - Feeder Warehouse,10138724,Saffola Crunchiez Ragi (Munchiez) Chips Masala...,1385,Marico Ltd.,5271.0,Marico Ltd,482,2026-04-30
5042,2468,Nagpur N1 - Feeder Warehouse,10161696,Kaya Everyday Cleansing Face Wipes30 piece - R...,1385,Marico Ltd.,5271.0,Marico Ltd,10,2026-04-30
5043,2468,Nagpur N1 - Feeder Warehouse,10000063,Saffola Masala & Coriander Oats(Pack)38 g - Rs 18,1385,Marico Ltd.,5271.0,Marico Ltd,1289,2026-04-30


In [150]:
blinkit_forecast_may = pd.read_csv('/data/aman_singh/acuuracy_check/Marico Ltd._forecast_May 2026_to_Aug 2026.csv')

blinkit_forecast_may.rename(columns = {'item_id':'item_code', 'May - forecast':'forecast_quantity'}, inplace = True)
blinkit_forecast_may.drop(columns = ['Aug - forecast', 'Jun - forecast', 'Jul - forecast'],inplace = True)
blinkit_forecast_may['date'] = '2026-05-31'
blinkit_forecast_may['date'] = pd.to_datetime(blinkit_forecast_may['date'])
blinkit_forecast_may

,facility_id,facility_name,item_code,item_name,manufacturer_id,manufacturer_name,vendor_id,vendor_name,forecast_quantity,date
0,1206,Super Store Lucknow L4 - Warehouse,10231034,Just Herbs Serum Infused Lip Gloss (04 Glimmer...,1385,Marico Ltd.,5271.0,Marico Ltd,0,2026-05-31
1,1206,Super Store Lucknow L4 - Warehouse,10044322,Parachute 100% Pure Coconut Oil(Pack)300 ml - ...,1385,Marico Ltd.,5271.0,Marico Ltd,2400,2026-05-31
2,1206,Super Store Lucknow L4 - Warehouse,10196027,Saffola Cuppa Flavoured Oats (Spicy Mexicana)(...,1385,Marico Ltd.,5271.0,Marico Ltd,61,2026-05-31
3,1206,Super Store Lucknow L4 - Warehouse,10265690,Saffola Soft & Creamy Rolled Oats(Jar)1 kg - R...,1385,Marico Ltd.,5271.0,Marico Ltd,39,2026-05-31
4,1206,Super Store Lucknow L4 - Warehouse,10010052,Revive Fabric Stiffener(Pack)400 gm - Rs 160.0,1385,Marico Ltd.,5271.0,Marico Ltd,828,2026-05-31
...,...,...,...,...,...,...,...,...,...,...
5401,4842,Kolkata K6 - Feeder Warehouse,10044322,Parachute 100% Pure Coconut Oil(Pack)300 ml - ...,1385,Marico Ltd.,5271.0,Marico Ltd,2210,2026-05-31
5402,4842,Kolkata K6 - Feeder Warehouse,10007074,Set Wet Casually Cool Styling Hair Gel 100 g(P...,1385,Marico Ltd.,5271.0,Marico Ltd,200,2026-05-31
5403,4842,Kolkata K6 - Feeder Warehouse,10112048,Saffola Pure Honey Active - Buy 1 Get 1 Free(B...,1385,Marico Ltd.,5271.0,Marico Ltd,88,2026-05-31
5404,4842,Kolkata K6 - Feeder Warehouse,10043648,Saffola Masala Peppy Tomato Oats(Pack)550 gm -...,1385,Marico Ltd.,5271.0,Marico Ltd,112,2026-05-31


In [151]:
blinkit_forecast_june = pd.read_csv('/data/aman_singh/acuuracy_check/Marico Ltd._forecast_Jun 2026_to_Sep 2026.csv')

blinkit_forecast_june.rename(columns = {'item_id':'item_code', 'Jun - forecast':'forecast_quantity'}, inplace = True)
blinkit_forecast_june.drop(columns = ['Aug - forecast', 'Sep - forecast', 'Jul - forecast'],inplace = True)
blinkit_forecast_june['date'] = '2026-06-30'
blinkit_forecast_june['date'] = pd.to_datetime(blinkit_forecast_june['date'])
blinkit_forecast_june

,facility_id,facility_name,item_code,item_name,manufacturer_id,manufacturer_name,vendor_id,vendor_name,forecast_quantity,date
0,3164,Mumbai M11 - Feeder Warehouse,10221228,Just Herbs 12 Chemical Free Nail Paint (Hot Re...,1385,Marico Ltd.,5271.0,Marico Ltd,24,2026-06-30
1,3164,Mumbai M11 - Feeder Warehouse,10000063,Saffola Masala & Coriander Oats(Pack)38 gm - R...,1385,Marico Ltd.,5271.0,Marico Ltd,1,2026-06-30
2,3164,Mumbai M11 - Feeder Warehouse,10000367,Saffola Masala Veggie Twist Oats(Pouch)38 gm -...,1385,Marico Ltd.,5271.0,Marico Ltd,4001,2026-06-30
3,3164,Mumbai M11 - Feeder Warehouse,10162271,Saffola Gold Unflavored Multigrain Oats (with ...,1385,Marico Ltd.,5271.0,Marico Ltd,15,2026-06-30
4,3164,Mumbai M11 - Feeder Warehouse,10010052,Revive Fabric Stiffener(Pack)400 gm - Rs 160.0,1385,Marico Ltd.,5271.0,Marico Ltd,665,2026-06-30
...,...,...,...,...,...,...,...,...,...,...
5836,5096,Faridabad - Feeder Warehouse,10221233,Just Herbs 12 Chemical Free Nail Paint (Gleami...,1385,Marico Ltd.,5271.0,Marico Ltd,10,2026-06-30
5837,5096,Faridabad - Feeder Warehouse,10241144,Just Herbs Ayurvedic Mini Lipstick Kit(Box)9.6...,1385,Marico Ltd.,5271.0,Marico Ltd,66,2026-06-30
5838,5096,Faridabad - Feeder Warehouse,10000910,Saffola Sunflower and Rice Bran Blended Cookin...,1385,Marico Ltd.,5271.0,Marico Ltd,1,2026-06-30
5839,5096,Faridabad - Feeder Warehouse,10147561,Just Herbs Party Ready Nail Paint Kit(Packet)1...,1385,Marico Ltd.,5271.0,Marico Ltd,11,2026-06-30


In [152]:
blinkit_unpivoted = pd.concat([blinkit_forecast_apr,blinkit_forecast_may,blinkit_forecast_june])
#blinkit_unpivoted = blinkit_forecast_apr.copy()
blinkit_unpivoted['chain_name'] = 'Blinkit'
blinkit_unpivoted

,facility_id,facility_name,item_code,item_name,manufacturer_id,manufacturer_name,vendor_id,vendor_name,forecast_quantity,date,chain_name
0,92,Super Store Dasna 2 - Warehouse,10224608,Parachute Advansed Gold Vitamin E Coconut Hair...,1385,Marico Ltd.,5271.0,Marico Ltd,134,2026-04-30,Blinkit
1,92,Super Store Dasna 2 - Warehouse,10010405,Parachute Advansed Aloe Vera Hair Oil for Soft...,1385,Marico Ltd.,5271.0,Marico Ltd,192,2026-04-30,Blinkit
2,92,Super Store Dasna 2 - Warehouse,10221231,Just Herbs 12 Chemical Free Nail Paint (Burgun...,1385,Marico Ltd.,5271.0,Marico Ltd,0,2026-04-30,Blinkit
3,92,Super Store Dasna 2 - Warehouse,10064182,Saffola Honey Active (500 g)(Pack)500 gm - Rs ...,1385,Marico Ltd.,5271.0,Marico Ltd,20,2026-04-30,Blinkit
4,92,Super Store Dasna 2 - Warehouse,10211385,Livon Nourishing Hair Serum(Bottle)100 ml - Rs...,1385,Marico Ltd.,5271.0,Marico Ltd,1,2026-04-30,Blinkit
...,...,...,...,...,...,...,...,...,...,...,...
5836,5096,Faridabad - Feeder Warehouse,10221233,Just Herbs 12 Chemical Free Nail Paint (Gleami...,1385,Marico Ltd.,5271.0,Marico Ltd,10,2026-06-30,Blinkit
5837,5096,Faridabad - Feeder Warehouse,10241144,Just Herbs Ayurvedic Mini Lipstick Kit(Box)9.6...,1385,Marico Ltd.,5271.0,Marico Ltd,66,2026-06-30,Blinkit
5838,5096,Faridabad - Feeder Warehouse,10000910,Saffola Sunflower and Rice Bran Blended Cookin...,1385,Marico Ltd.,5271.0,Marico Ltd,1,2026-06-30,Blinkit
5839,5096,Faridabad - Feeder Warehouse,10147561,Just Herbs Party Ready Nail Paint Kit(Packet)1...,1385,Marico Ltd.,5271.0,Marico Ltd,11,2026-06-30,Blinkit


In [153]:
swiggy_forecast_apr = pd.read_excel('/data/aman_singh/acuuracy_check/MARICO LIMITED_swiggy_apr.xlsx')
swiggy_forecast_apr.columns = swiggy_forecast_apr.columns.str.lower()
swiggy_forecast_apr.rename(columns = {'wh_name':'facility_name', 'apr_buy_qty':'forecast_quantity'}, inplace = True)
swiggy_forecast_apr.drop(columns = ['may_buy_qty', 'jun_buy_qty'],inplace = True)
swiggy_forecast_apr['date'] = '2026-04-30'
swiggy_forecast_apr['date'] = pd.to_datetime(swiggy_forecast_apr['date'])
swiggy_forecast_apr

,item_code,sku_name,company,brand,city,facility_name,forecast_quantity,date
0,42915,"Saffola Honey Gold, 100% Pure Honey, Made With...",MARICO LIMITED,Saffola,GUWAHATI,GAU IM1,6,2026-04-30
1,44288,Kaya Refreshing Mattifying Wipes | Vitamin E &...,MARICO LIMITED,Kaya,BANGALORE,BLR ECOM2,120,2026-04-30
2,44288,Kaya Refreshing Mattifying Wipes | Vitamin E &...,MARICO LIMITED,Kaya,MUMBAI,MUM FC22,0,2026-04-30
3,44733,Kaya Deep Nourish Elbow & Foot Cream with Shea...,MARICO LIMITED,Kaya,JAIPUR,JAI IM1,0,2026-04-30
4,47441,Just Herbs Spf 20+ Tinted Lip Balm (Cherry),MARICO LIMITED,Just Herbs,CHENNAI,CHE AMB IM2,304,2026-04-30
...,...,...,...,...,...,...,...,...
7824,978064,Just Herbs Scarlet Maroon Long Stay Smudge and...,MARICO LIMITED,Just Herbs,AHMEDABAD,AHM DELHIVERY,0,2026-04-30
7825,978064,Just Herbs Scarlet Maroon Long Stay Smudge and...,MARICO LIMITED,Just Herbs,KOCHI,KOC IM1,24,2026-04-30
7826,987028,Parachute Advansed Pro Hair Growth Serum | Red...,MARICO LIMITED,Parachute Advansed,HYDERABAD,HYD IM1,0,2026-04-30
7827,991861,Kaya Sea Salt Exfoliating Shower Gel,MARICO LIMITED,Kaya,JAIPUR,JAI IM1,24,2026-04-30


In [154]:
swiggy_forecast_may = pd.read_excel('/data/aman_singh/acuuracy_check/MARICO LIMITED_swiggy_may.xlsx')
swiggy_forecast_may.columns = swiggy_forecast_may.columns.str.lower()
swiggy_forecast_may.rename(columns = {'wh_name':'facility_name', 'may_buy_qty':'forecast_quantity'}, inplace = True)
swiggy_forecast_may.drop(columns = ['jul_buy_qty', 'jun_buy_qty'],inplace = True)
swiggy_forecast_may['date'] = '2026-05-31'
swiggy_forecast_may['date'] = pd.to_datetime(swiggy_forecast_may['date'])
swiggy_forecast_may

,item_code,sku_name,company,brand,city,facility_name,forecast_quantity,date
0,57794,"Saffola Oats with Nutty Chocolate, Healthy & T...",MARICO LIMITED,Saffola,KOLKATA,KOLKATA ECOM,0,2026-05-31
1,60103,"Parachute Advansed Coconut Baby Soap (Soft, Mo...",MARICO LIMITED,Parachute,GUWAHATI,GAU IM1,60,2026-05-31
2,60129,Just Herbs Mint + Activated Charcoal Cooling B...,MARICO LIMITED,Just Herbs,GURGAON,DLHY GGNFC5,0,2026-05-31
3,60129,Just Herbs Mint + Activated Charcoal Cooling B...,MARICO LIMITED,Just Herbs,VIZAG,VIZ IM1,60,2026-05-31
4,60504,"Parachute Advansed Baby Powder (Soft, Fresh skin)",MARICO LIMITED,Parachute Advansed,KOCHI,KOC IM1,48,2026-05-31
...,...,...,...,...,...,...,...,...
8303,987028,Parachute Advansed Pro Hair Growth Serum | Red...,MARICO LIMITED,Parachute Advansed,PUNE,PUN DELHIVERY,0,2026-05-31
8304,991861,Kaya Sea Salt Exfoliating Shower Gel,MARICO LIMITED,Kaya,BANGALORE,BLR IM4,0,2026-05-31
8305,991861,Kaya Sea Salt Exfoliating Shower Gel,MARICO LIMITED,Kaya,NOIDA,NOI IM1,24,2026-05-31
8306,999977,Just Herbs Serum Foundation For Face Makeup Wi...,MARICO LIMITED,Just Herbs,BANGALORE,BLR ECOM2,0,2026-05-31


In [155]:
swiggy_forecast_jun = pd.read_excel('/data/aman_singh/acuuracy_check/MARICO LIMITED_swiggy_june.xlsx')
swiggy_forecast_jun.columns = swiggy_forecast_jun.columns.str.lower()
swiggy_forecast_jun.rename(columns = {'wh_name':'facility_name', 'jun_buy_qty':'forecast_quantity'}, inplace = True)
swiggy_forecast_jun.drop(columns = ['jul_buy_qty', 'aug_buy_qty'],inplace = True)
swiggy_forecast_jun['date'] = '2026-06-30'
swiggy_forecast_jun['date'] = pd.to_datetime(swiggy_forecast_jun['date'])
swiggy_forecast_jun

,item_code,sku_name,company,brand,city,facility_name,forecast_quantity,date
0,10104,Parachute Advansed Ayurvedic Hot Oil warming C...,MARICO LIMITED,Parachute Advansed,HYDERABAD,HYD IM2,0,2026-06-30
1,10589,Saffola Total Refined Cooking Oil Blend Of Ric...,MARICO LIMITED,Saffola,GURGAON,DLHY GGNFC5,0,2026-06-30
2,10589,Saffola Total Refined Cooking Oil Blend Of Ric...,MARICO LIMITED,Saffola,VIZAG,VIZ IM1,0,2026-06-30
3,10616,"Parachute Advansed Deep Nourish Body Lotion,Wi...",MARICO LIMITED,Parachute,BANGALORE,BLR IM4,120,2026-06-30
4,10616,"Parachute Advansed Deep Nourish Body Lotion,Wi...",MARICO LIMITED,Parachute,MUMBAI,MUM FC22,0,2026-06-30
...,...,...,...,...,...,...,...,...
8364,987028,Parachute Advansed Pro Hair Growth Serum | Red...,MARICO LIMITED,Parachute Advansed,CHENNAI,CHN ECOM,20,2026-06-30
8365,987028,Parachute Advansed Pro Hair Growth Serum | Red...,MARICO LIMITED,Parachute Advansed,VIZAG,VIZ IM1,20,2026-06-30
8366,991861,Kaya Sea Salt Exfoliating Shower Gel,MARICO LIMITED,Kaya,HYDERABAD,HYD IM4,24,2026-06-30
8367,999977,Just Herbs Serum Foundation For Face Makeup Wi...,MARICO LIMITED,Just Herbs,BANGALORE,BLR IM4,1,2026-06-30


In [156]:
Swiggy_unpivoted = pd.concat([swiggy_forecast_apr,swiggy_forecast_may,swiggy_forecast_jun])
#Swiggy_unpivoted = swiggy_forecast_apr.copy()
Swiggy_unpivoted['chain_name'] = 'Swiggy'
Swiggy_unpivoted

,item_code,sku_name,company,brand,city,facility_name,forecast_quantity,date,chain_name
0,42915,"Saffola Honey Gold, 100% Pure Honey, Made With...",MARICO LIMITED,Saffola,GUWAHATI,GAU IM1,6,2026-04-30,Swiggy
1,44288,Kaya Refreshing Mattifying Wipes | Vitamin E &...,MARICO LIMITED,Kaya,BANGALORE,BLR ECOM2,120,2026-04-30,Swiggy
2,44288,Kaya Refreshing Mattifying Wipes | Vitamin E &...,MARICO LIMITED,Kaya,MUMBAI,MUM FC22,0,2026-04-30,Swiggy
3,44733,Kaya Deep Nourish Elbow & Foot Cream with Shea...,MARICO LIMITED,Kaya,JAIPUR,JAI IM1,0,2026-04-30,Swiggy
4,47441,Just Herbs Spf 20+ Tinted Lip Balm (Cherry),MARICO LIMITED,Just Herbs,CHENNAI,CHE AMB IM2,304,2026-04-30,Swiggy
...,...,...,...,...,...,...,...,...,...
8364,987028,Parachute Advansed Pro Hair Growth Serum | Red...,MARICO LIMITED,Parachute Advansed,CHENNAI,CHN ECOM,20,2026-06-30,Swiggy
8365,987028,Parachute Advansed Pro Hair Growth Serum | Red...,MARICO LIMITED,Parachute Advansed,VIZAG,VIZ IM1,20,2026-06-30,Swiggy
8366,991861,Kaya Sea Salt Exfoliating Shower Gel,MARICO LIMITED,Kaya,HYDERABAD,HYD IM4,24,2026-06-30,Swiggy
8367,999977,Just Herbs Serum Foundation For Face Makeup Wi...,MARICO LIMITED,Just Herbs,BANGALORE,BLR IM4,1,2026-06-30,Swiggy


In [157]:
blinkit_unpivoted = blinkit_unpivoted.groupby(['chain_name','facility_name','item_code','date'])['forecast_quantity'].sum().reset_index()
swiggy_unpivoted = Swiggy_unpivoted.groupby(['chain_name','facility_name','item_code','date'])['forecast_quantity'].sum().reset_index()


In [158]:
chain_forecast_unpivoted = pd.concat([blinkit_unpivoted,swiggy_unpivoted])
chain_forecast_unpivoted

,chain_name,facility_name,item_code,date,forecast_quantity
0,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000059,2026-04-30,313
1,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000059,2026-05-31,341
2,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000059,2026-06-30,199
3,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000362,2026-04-30,139
4,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000362,2026-05-31,131
...,...,...,...,...,...
24501,Swiggy,VIZ IM1,987028,2026-05-31,40
24502,Swiggy,VIZ IM1,987028,2026-06-30,20
24503,Swiggy,VIZ IM1,999977,2026-04-30,0
24504,Swiggy,VIZ IM1,999977,2026-05-31,3


In [159]:
mapping = pd.read_excel("/data/aman_singh/mt_forecast/Daily Offtake Tracker - Jun'26.xlsb",sheet_name = 'Mapping')


In [160]:
len_before_merge = len(chain_forecast_unpivoted)
chain_forecast_unpivoted['item_code'] = chain_forecast_unpivoted['item_code'].astype(str)
mapping['asin'] = mapping['asin'].astype(str)
temp = mapping[['platform_name','asin','EAN','PSKU','UOM','Vol per unit']].drop_duplicates()

temp = temp[temp['platform_name'].isin(['Blinkit', 'Swiggy', 'Zepto'])]
#temp['platform_name'].unique()
duplicates = temp[temp.duplicated(subset="asin", keep=False)]
duplicates



,platform_name,asin,EAN,PSKU,UOM,Vol per unit


In [161]:
temp['PSKU'] = temp['PSKU'].astype(str)
temp['EAN'] = temp['EAN'].astype(str)
temp['UOM'] = temp['UOM'].astype(str)
len_before_merge = len(chain_forecast_unpivoted)
df_chk = chain_forecast_unpivoted.merge(temp,
                  left_on = ['item_code'], right_on = ['asin'], how = 'left')
assert(len_before_merge == len(df_chk))
df_chk['date'] = pd.to_datetime(df_chk['date'])

In [162]:
df_chk

,chain_name,facility_name,item_code,date,forecast_quantity,platform_name,asin,EAN,PSKU,UOM,Vol per unit
0,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000059,2026-04-30,313,Blinkit,10000059,8901088000772,718322,KL,5000.0
1,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000059,2026-05-31,341,Blinkit,10000059,8901088000772,718322,KL,5000.0
2,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000059,2026-06-30,199,Blinkit,10000059,8901088000772,718322,KL,5000.0
3,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000362,2026-04-30,139,Blinkit,10000362,8901088002530,718328,KL,900.0
4,Blinkit,Ahmedabad A2 - Feeder Warehouse,10000362,2026-05-31,131,Blinkit,10000362,8901088002530,718328,KL,900.0
...,...,...,...,...,...,...,...,...,...,...,...
40793,Swiggy,VIZ IM1,987028,2026-05-31,40,Swiggy,987028,8901088732864,732011,L,50.0
40794,Swiggy,VIZ IM1,987028,2026-06-30,20,Swiggy,987028,8901088732864,732011,L,50.0
40795,Swiggy,VIZ IM1,999977,2026-04-30,0,NaN,NaN,NaN,NaN,NaN,NaN
40796,Swiggy,VIZ IM1,999977,2026-05-31,3,NaN,NaN,NaN,NaN,NaN,NaN


In [163]:
duplicates = df_chk[df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False)]
duplicates.isnull().sum()

chain_name               0
facility_name            0
item_code                0
date                     0
forecast_quantity        0
platform_name        10106
asin                 10106
EAN                  10106
PSKU                 10106
UOM                  10106
Vol per unit         10106
dtype: int64

In [164]:
df_chk[(df_chk['PSKU'].isna()) & (df_chk['chain_name'] == 'Swiggy')]#['forecast_quantity'].sum()

,chain_name,facility_name,item_code,date,forecast_quantity,platform_name,asin,EAN,PSKU,UOM,Vol per unit
16504,Swiggy,AHM DELHIVERY,47441,2026-04-30,0,NaN,NaN,NaN,NaN,NaN,NaN
16505,Swiggy,AHM DELHIVERY,47441,2026-05-31,304,NaN,NaN,NaN,NaN,NaN,NaN
16506,Swiggy,AHM DELHIVERY,47441,2026-06-30,0,NaN,NaN,NaN,NaN,NaN,NaN
16507,Swiggy,AHM DELHIVERY,50708,2026-04-30,0,NaN,NaN,NaN,NaN,NaN,NaN
16508,Swiggy,AHM DELHIVERY,50708,2026-05-31,0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
40788,Swiggy,VIZ IM1,978064,2026-05-31,0,NaN,NaN,NaN,NaN,NaN,NaN
40789,Swiggy,VIZ IM1,978064,2026-06-30,24,NaN,NaN,NaN,NaN,NaN,NaN
40795,Swiggy,VIZ IM1,999977,2026-04-30,0,NaN,NaN,NaN,NaN,NaN,NaN
40796,Swiggy,VIZ IM1,999977,2026-05-31,3,NaN,NaN,NaN,NaN,NaN,NaN


In [165]:
df_chk[(df_chk['chain_name'] == 'Swiggy')]#['forecast_quantity'].sum()

,chain_name,facility_name,item_code,date,forecast_quantity,platform_name,asin,EAN,PSKU,UOM,Vol per unit
16292,Swiggy,AHM DELHIVERY,3,2026-04-30,192,Swiggy,3,89002940,718299,KL,100.0
16293,Swiggy,AHM DELHIVERY,3,2026-05-31,384,Swiggy,3,89002940,718299,KL,100.0
16294,Swiggy,AHM DELHIVERY,3,2026-06-30,0,Swiggy,3,89002940,718299,KL,100.0
16295,Swiggy,AHM DELHIVERY,102,2026-04-30,0,Swiggy,102,8901088002530,718328,KL,900.0
16296,Swiggy,AHM DELHIVERY,102,2026-05-31,260,Swiggy,102,8901088002530,718328,KL,900.0
...,...,...,...,...,...,...,...,...,...,...,...
40793,Swiggy,VIZ IM1,987028,2026-05-31,40,Swiggy,987028,8901088732864,732011,L,50.0
40794,Swiggy,VIZ IM1,987028,2026-06-30,20,Swiggy,987028,8901088732864,732011,L,50.0
40795,Swiggy,VIZ IM1,999977,2026-04-30,0,NaN,NaN,NaN,NaN,NaN,NaN
40796,Swiggy,VIZ IM1,999977,2026-05-31,3,NaN,NaN,NaN,NaN,NaN,NaN


In [166]:
df_chk = df_chk.dropna(subset = ['PSKU'])
df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False).sum()

458

In [167]:
# duplicates = df_chk[df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False)]
# duplicates.sort_values(by=['chain_name','facility_name','PSKU','date'])[:60]

In [168]:
# duplicates.sort_values(by=['chain_name','facility_name','PSKU','date']).to_csv('duplicates_swiggy2.csv')

In [169]:
# df_chk = df_chk.sort_values('forecast_quantity', ascending=False) \
#        .drop_duplicates(subset=['chain_name', 'facility_name','PSKU' , 'date'], keep='first')
# df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False).sum()

In [170]:
df_chk.columns

Index(['chain_name', 'facility_name', 'item_code', 'date', 'forecast_quantity',
       'platform_name', 'asin', 'EAN', 'PSKU', 'UOM', 'Vol per unit'],
      dtype='object')

In [171]:
df_chk['month_date'] = df_chk['date'] + pd.offsets.MonthEnd(0)

df_chk.rename(columns = {'item_code':'platform_code', 'EAN':'eancode', 'UOM':'uom_reporting',
                         'Vol per unit':'vol_per_unit'},inplace=True)
df_chk['vol_in_lit'] = df_chk['forecast_quantity']*df_chk['vol_per_unit']/1000
df_chk['vol_in_rum'] = df_chk.apply(
    lambda x: x['vol_in_lit'] / 1000 if x['uom_reporting'] in ['KL', 'TO'] else x['vol_in_lit'],
    axis=1
)

df_chk = df_chk.groupby(['chain_name','facility_name', 'PSKU','month_date'])[['vol_in_rum']].sum().reset_index()
df_chk['PSKU'] = df_chk['PSKU'].astype(int)
df_chk.rename(columns = {'PSKU':'parent_material_code'}, inplace = True)
df_chk

,chain_name,facility_name,parent_material_code,month_date,vol_in_rum
0,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-04-30,9.516
1,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-05-31,4.728
2,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-06-30,5.358
3,Blinkit,Ahmedabad A2 - Feeder Warehouse,718312,2026-04-30,0.417
4,Blinkit,Ahmedabad A2 - Feeder Warehouse,718312,2026-05-31,0.323
...,...,...,...,...,...
30458,Swiggy,VIZ IM1,810685,2026-05-31,0.008
30459,Swiggy,VIZ IM1,810685,2026-06-30,0.008
30460,Swiggy,VIZ IM1,810738,2026-04-30,17.376
30461,Swiggy,VIZ IM1,810738,2026-05-31,17.376


In [172]:
df_chk.duplicated(subset=['chain_name','facility_name','parent_material_code','month_date'], keep=False).sum()

0

In [173]:
facility_to_city_mappings_df = pd.read_excel(
    r'/data/aman_singh/mt_forecast/City Mappings QCOM.xlsb', 
    'Sheet1'
)
facility_to_city_mappings_df.columns = facility_to_city_mappings_df.columns.str.lower()
facility_to_city_mappings_df.columns = ['chain', 'facility_name', 'city', 'customer', 'marico_depot',
       'status', 'depot_name']
facility_to_city_mappings_df = facility_to_city_mappings_df[
    facility_to_city_mappings_df['customer'].notna()
]
facility_to_city_mappings_df['facility_name'] = facility_to_city_mappings_df['facility_name'].str.lower()
facility_to_city_mappings_df['city'] = facility_to_city_mappings_df['city'].str.lower()
facility_to_city_mappings_df.head()
customer_depot_mappings_df = pd.read_sql("""
SELECT DISTINCT customer_code, depot_code, channel_name 
FROM mst_customer
WHERE company_code='MIL' AND
    latest_record_ind=1
ORDER BY 3, 1, 2
""",
prod_conn
)
customer_depot_mappings_df.columns = customer_depot_mappings_df.columns.str.lower()
customer_depot_mappings_df.duplicated(subset=['customer_code']).sum()
customer_depot_mappings_df['customer_code'] = customer_depot_mappings_df['customer_code'].astype(str)
facility_to_city_mappings_df['customer'] = facility_to_city_mappings_df['customer'].astype(str)
customer_depot_mappings_df.dtypes
facility_to_city_mappings_df.dtypes
len_before_merge = len(facility_to_city_mappings_df)
facility_to_city_mappings_df = facility_to_city_mappings_df.merge(
    customer_depot_mappings_df[['customer_code', 'depot_code']].drop_duplicates().rename(
        columns={'customer_code': 'customer'}
    ),
    on=['customer'],
    how='left'
)
assert len_before_merge == len(facility_to_city_mappings_df)
del len_before_merge

In [174]:
facility_to_city_mappings_df#.isnull().sum()

,chain,facility_name,city,customer,marico_depot,status,depot_name,depot_code
0,Blinkit,farukhnagar f2 - feeder warehouse,gurugram,16012,D115,Active,NaN,D115
1,Blinkit,ahmedabad a2 - feeder warehouse,ahmedabad,15616,D354,Active,NaN,D354
2,Blinkit,hyderabad h3 - feeder warehouse,hyderabad,16005,D530,Active,NaN,D530
3,Blinkit,lucknow l5 - feeder warehouse,lucknow,15855,D113,Active,NaN,D113
4,Blinkit,super store hyderabad h2 - warehouse,hyderabad,9895,D530,Active,NaN,D530
...,...,...,...,...,...,...,...,...
144,Zepto,lko-dry-mh-sohramau,lucknow,16804,D113,Active,NaN,D113
145,Blinkit,hot mumbai m12 - feeder,mumbai,18603,D356,Active,NaN,D356
146,Blinkit,hot patna p2 - feeder,patna,18604,D233,Active,NaN,D233
147,Swiggy,scootsy logistics private limited- coimbatore,coimbatore,18366,D676,Active,NaN,D676


In [175]:
facility_to_city_mappings_df.rename(columns = {'facility_name':'FC', 'chain':'chain_name'}, inplace = True)
facility_to_city_mappings_df['FC'] = facility_to_city_mappings_df['FC'].str.lower()

facility_to_city_mappings_df[facility_to_city_mappings_df.duplicated(subset = ['chain_name','FC'],keep=False)]
facility_to_city_mappings_df = facility_to_city_mappings_df[['chain_name','FC','depot_code']].drop_duplicates()


In [176]:
df_chk.rename(columns = {'facility_name':'FC'},inplace = True)
df_chk['FC'] = df_chk['FC'].str.lower()
df_chk

,chain_name,FC,parent_material_code,month_date,vol_in_rum
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-04-30,9.516
1,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-05-31,4.728
2,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-06-30,5.358
3,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-04-30,0.417
4,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-05-31,0.323
...,...,...,...,...,...
30458,Swiggy,viz im1,810685,2026-05-31,0.008
30459,Swiggy,viz im1,810685,2026-06-30,0.008
30460,Swiggy,viz im1,810738,2026-04-30,17.376
30461,Swiggy,viz im1,810738,2026-05-31,17.376


In [177]:
xy = df_chk.copy()

In [178]:
df_chk = df_chk.merge(facility_to_city_mappings_df, on = ['chain_name','FC'], how = 'left')
df_chk#.isnull().sum()

,chain_name,FC,parent_material_code,month_date,vol_in_rum,depot_code
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-04-30,9.516,D354
1,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-05-31,4.728,D354
2,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-06-30,5.358,D354
3,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-04-30,0.417,D354
4,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-05-31,0.323,D354
...,...,...,...,...,...,...
30458,Swiggy,viz im1,810685,2026-05-31,0.008,D572
30459,Swiggy,viz im1,810685,2026-06-30,0.008,D572
30460,Swiggy,viz im1,810738,2026-04-30,17.376,D572
30461,Swiggy,viz im1,810738,2026-05-31,17.376,D572


In [179]:
df_chk[df_chk['depot_code'].isna()]['vol_in_rum'].sum()/df_chk['vol_in_rum'].sum()

0.030205128656112804

In [180]:
df_chk[df_chk['depot_code'].isna()][['chain_name','FC']].drop_duplicates()

,chain_name,FC
8733,Blinkit,mumbai m12 - feeder warehouse
9937,Blinkit,patna p2 - feeder warehouse
10419,Blinkit,pune p3 - feeder warehouse
16045,Swiggy,blr im4


In [181]:
df_chk[df_chk['chain_name'] == 'Swiggy']['vol_in_rum'].sum()

70812.093478

In [182]:
df_chk

,chain_name,FC,parent_material_code,month_date,vol_in_rum,depot_code
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-04-30,9.516,D354
1,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-05-31,4.728,D354
2,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-06-30,5.358,D354
3,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-04-30,0.417,D354
4,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-05-31,0.323,D354
...,...,...,...,...,...,...
30458,Swiggy,viz im1,810685,2026-05-31,0.008,D572
30459,Swiggy,viz im1,810685,2026-06-30,0.008,D572
30460,Swiggy,viz im1,810738,2026-04-30,17.376,D572
30461,Swiggy,viz im1,810738,2026-05-31,17.376,D572


In [183]:
material_master_df = pd.read_sql(
    """select * from mst_material 
    where latest_record_ind=1 and company_code='MIL'""",
    prod_conn
)
material_master_df.columns = material_master_df.columns.str.lower()
assert material_master_df.duplicated(
    subset=['company_code', 'material_code']).sum() == 0
material_master_df = material_master_df.rename(columns=
    {'material_group_code': 'brand_code'})
material_master_df['material_code'] = material_master_df['material_code'].astype(np.int64)
material_master_df.duplicated(subset=['material_code', 'parent_material_code', 'brand_code']).sum()

0

In [184]:
# material_master_df[['parent_material_code', 'brand_code']].dtypes
material_master_df['parent_material_code'] = material_master_df['parent_material_code'].astype(int)
material_master_df.loc[material_master_df['parent_material_code'].isin([725930,731857]), 'brand_code'] = 'H&C_ALMND'

# offtake_df.drop(columns = ['brand_code'],inplace = True)
len_before_merge = len(df_chk)
df_chk = df_chk.merge(
    material_master_df[['parent_material_code', 'brand_code']].drop_duplicates(),
    left_on=['parent_material_code'],right_on = ['parent_material_code'],
    how='left'
)
assert len_before_merge == len(df_chk)

In [185]:
df_chk

,chain_name,FC,parent_material_code,month_date,vol_in_rum,depot_code,brand_code
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-04-30,9.516,D354,SAFF GOLD
1,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-05-31,4.728,D354,SAFF GOLD
2,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-06-30,5.358,D354,SAFF GOLD
3,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-04-30,0.417,D354,PCNO(R)
4,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-05-31,0.323,D354,PCNO(R)
...,...,...,...,...,...,...,...
30458,Swiggy,viz im1,810685,2026-05-31,0.008,D572,SAF-MUSLI
30459,Swiggy,viz im1,810685,2026-06-30,0.008,D572,SAF-MUSLI
30460,Swiggy,viz im1,810738,2026-04-30,17.376,D572,PABABY_GM
30461,Swiggy,viz im1,810738,2026-05-31,17.376,D572,PABABY_GM


In [186]:
def read_qtr_ind_rate_table():
    """
    Fetch the club sku information from  DWH_SAP_INDEX_TURNOVER_MONTHWISE table.

    Return:
        qtr_ind_rate_data: pandas dataframe
        - dataframe contains all the results from the index rate table.
    """
    connection = get_dbconnection(db_name='PROD')
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=connection, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    connection.close()
    return qtr_ind_rate



In [187]:
qtr_df = read_qtr_ind_rate_table()
qtr_df.columns = qtr_df.columns.str.lower()
qtr_df.head()


len_before_merge = len(df_chk)

df_chk = df_chk.rename(columns={'material_group_code': 'brand_code'}).merge(
    qtr_df.drop('month_date', axis=1),
    on=['brand_code'],
    how='left'
)

assert len_before_merge == len(df_chk)


Credentials retrieved successfully for prod db.


In [188]:
df_chk['value'] = df_chk['vol_in_rum']*df_chk['qtr_ind_rate']/10**7
df_chk[df_chk['chain_name'] == 'Swiggy'].groupby(['month_date'])['value'].sum()

month_date
2026-04-30    5.116882
2026-05-31    5.288487
2026-06-30    5.062593
Name: value, dtype: float64

In [194]:
df_chk[(df_chk['depot_code'].isna()) & (df_chk['chain_name'] == 'Swiggy')].groupby(['month_date'])['value'].sum()

month_date
2026-05-31    0.125483
2026-06-30    0.170864
Name: value, dtype: float64

In [40]:
chain_forecast_zepto = pd.read_csv('/data/aman_singh/acuuracy_check/Marico_Limited_projection_apr_zepto.csv')
chain_forecast_zepto.columns = chain_forecast_zepto.columns.str.lower()
chain_forecast_zepto

,month,cluster_dry,product_variant_id,product_name,category_name,subcategory_name,l3_category_name,brand_name,manufacturer,packsize,unit_of_measure,unit_mrp,projected_qty
0,June,Pune,78579bf1-1e88-4094-8181-d3a2ced0b045,Saffola Active Rice Bran & Soyabean Oil | Rich...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Saffola,Marico Limited,4250.0,GRAM,822.0,489.0
1,May,Hyderabad,021c7b96-7a79-492b-9226-f74c9e5ce670,Parachute Advansed Biotin & Coconut Hair Oil |...,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,300.0,MILLILITRE,275.0,504.0
2,April,Bengaluru,107f0406-e2a5-41cb-a072-bd0ee952a202,Parachute Advansed Soft Touch Body Lotion With...,Skincare,Body Lotion & Moisturizer,Body Lotion,Parachute,Marico Limited,400.0,MILLILITRE,385.0,716.0
3,April,SAS Nagar,26c1f539-8aea-45ec-86d8-21a6a42a5d67,Parachute Advansed Deep Nourish Body Lotion Wi...,Skincare,Body Lotion & Moisturizer,Body Lotion,Parachute,Marico Limited,225.0,MILLILITRE,240.0,487.0
4,April,Chennai,91fc69e9-a256-4477-a501-ae56ec51a123,"Parachute Advansed Coconut Baby Soap | Soft, M...",Baby Care,Baby Bath,Baby Soap,Parachute,Marico Limited,75.0,GRAM,157.0,486.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5303,April,NCR,709bd327-baf4-4104-a46d-fdba2b80b99c,Parachute 100 % Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,300.0,MILLILITRE,124.0,12010.0
5304,May,Chennai,709bd327-baf4-4104-a46d-fdba2b80b99c,Parachute 100 % Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,300.0,MILLILITRE,124.0,12134.0
5305,June,Bengaluru,e2eeab46-1109-41ae-b90b-a9c3a429e62e,"Saffola Gold Oil, Power of 3, Pouch","Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Saffola,Marico Limited,910.0,GRAM,186.0,11559.0
5306,June,Ahmedabad,aaa43e9e-89dc-4c74-acc9-1aa50c57ff1c,Saffola Active Rice Bran & Soyabean Oil | Rich...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Saffola,Marico Limited,850.0,GRAM,163.0,11066.0


In [41]:
chain_forecast_zepto['month'].unique()

array(['June', 'May', 'April'], dtype=object)

In [42]:

month_map = {
    'May': '2026-05-31',
    'June': '2026-06-30',
    'April': '2026-04-30'
}

# Apply mapping
chain_forecast_zepto['date'] = chain_forecast_zepto['month'].map(month_map)
chain_forecast_zepto['date'] = pd.to_datetime(chain_forecast_zepto['date'])
chain_forecast_zepto_april = chain_forecast_zepto[chain_forecast_zepto['date'] == '2026-04-30']
chain_forecast_zepto_april

,month,cluster_dry,product_variant_id,product_name,category_name,subcategory_name,l3_category_name,brand_name,manufacturer,packsize,unit_of_measure,unit_mrp,projected_qty,date
2,April,Bengaluru,107f0406-e2a5-41cb-a072-bd0ee952a202,Parachute Advansed Soft Touch Body Lotion With...,Skincare,Body Lotion & Moisturizer,Body Lotion,Parachute,Marico Limited,400.0,MILLILITRE,385.0,716.0,2026-04-30
3,April,SAS Nagar,26c1f539-8aea-45ec-86d8-21a6a42a5d67,Parachute Advansed Deep Nourish Body Lotion Wi...,Skincare,Body Lotion & Moisturizer,Body Lotion,Parachute,Marico Limited,225.0,MILLILITRE,240.0,487.0,2026-04-30
4,April,Chennai,91fc69e9-a256-4477-a501-ae56ec51a123,"Parachute Advansed Coconut Baby Soap | Soft, M...",Baby Care,Baby Bath,Baby Soap,Parachute,Marico Limited,75.0,GRAM,157.0,486.0,2026-04-30
7,April,SAS Nagar,4d9913da-86d4-4520-9985-1948e50ac1c5,Parachute Advansed Deep Nourish Body Lotion Wi...,Skincare,Body Lotion & Moisturizer,Body Lotion,Parachute,Marico Limited,400.0,MILLILITRE,385.0,482.0,2026-04-30
8,April,Mumbai,91fc69e9-a256-4477-a501-ae56ec51a123,"Parachute Advansed Coconut Baby Soap | Soft, M...",Baby Care,Baby Bath,Baby Soap,Parachute,Marico Limited,75.0,GRAM,157.0,482.0,2026-04-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5294,April,Pune,aaa43e9e-89dc-4c74-acc9-1aa50c57ff1c,Saffola Active Rice Bran & Soyabean Oil | Rich...,"Atta, Rice, Oil & Dals",Oil,Vegetable Oil,Saffola,Marico Limited,850.0,GRAM,163.0,13421.0,2026-04-30
5298,April,Pune,e011578a-374b-406a-890d-f092005c203d,Saffola Masala Oats |Classic Masala | Anytime ...,Breakfast & Sauces,Muesli & Oats,Oats,Saffola Foods,Marico Limited,38.0,GRAM,17.0,12775.0,2026-04-30
5299,April,Bengaluru,e011578a-374b-406a-890d-f092005c203d,Saffola Masala Oats |Classic Masala | Anytime ...,Breakfast & Sauces,Muesli & Oats,Oats,Saffola Foods,Marico Limited,38.0,GRAM,17.0,12741.0,2026-04-30
5300,April,Lucknow,e011578a-374b-406a-890d-f092005c203d,Saffola Masala Oats |Classic Masala | Anytime ...,Breakfast & Sauces,Muesli & Oats,Oats,Saffola Foods,Marico Limited,38.0,GRAM,17.0,12725.0,2026-04-30


In [43]:
chain_forecast_zepto = pd.read_csv('/data/aman_singh/acuuracy_check/Marico_Limited_projection_jun_zepto.csv')
chain_forecast_zepto.columns = chain_forecast_zepto.columns.str.lower()
chain_forecast_zepto
chain_forecast_zepto['month'].unique()

month_map = {
    'Jun': '2026-06-30',
    'May': '2026-05-31',
    'Jul': '2026-07-31'
}

# Apply mapping
chain_forecast_zepto['date'] = chain_forecast_zepto['month'].map(month_map)
chain_forecast_zepto['date'] = pd.to_datetime(chain_forecast_zepto['date'])
chain_forecast_zepto_may = chain_forecast_zepto[chain_forecast_zepto['date'].isin(['2026-05-31'])]
chain_forecast_zepto_may

,month,cluster_dry,product_variant_id,product_name,category_name,subcategory_name,l3_category_name,brand_name,manufacturer,packsize,unit_of_measure,unit_mrp,projected_qty,date
0,May,Jaipur,f1b1c703-46a6-43e2-8b49-43c6a193a2a2,Bio Oil Original Skincare Oil Suitable For Str...,Skincare,Body Care,Body Oil,Bio Oil,Marico Limited,60.0,MILLILITRE,550.0,20.0,2026-05-31
3,May,SAS Nagar,a74eaad2-e93b-46d3-8e6d-d11a56d38632,Parachute Advansed Baby Wipes with virgin coco...,Baby Care,Baby Wipes,Baby Wipes,Parachute,Marico Limited,1.0,PIECE,230.0,12.0,2026-05-31
8,May,Mumbai,a0841095-9583-4788-8c67-cbe41ddab0d7,Just Herbs Eyeliner | Multicolour | Waterproof,Makeup & Beauty,Eye Makeup,Eye Liner,Justherbs,Marico Limited,42.0,GRAM,599.0,60.0,2026-05-31
10,May,Chennai,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,261.0,2104.0,2026-05-31
12,May,Ahmedabad,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,261.0,1104.0,2026-05-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5579,May,Jaipur,f702765e-ea47-41a5-90e8-3c797e6b29fe,Saffola 25g High Protein Oats | 14g Fibre | No...,Breakfast & Sauces,Muesli & Oats,Oats,Saffola Foods,Marico Limited,400.0,GRAM,299.0,28.0,2026-05-31
5581,May,Pune,03843645-8523-4571-8c92-7b3b5d292930,Livon Style Pro Keratin Serum 10X Stronger & S...,Hair Care,Hair Serum & Polish,Hair Serum,Livon,Marico Limited,100.0,MILLILITRE,665.0,12.0,2026-05-31
5583,May,Kolkata,da954c80-83a5-43f0-8c50-2354cd689945,Just Herbs Hair Growth Oil With Rosemary And C...,Hair Care,Hair Oil,Hair Growth Oil,Justherbs,Marico Limited,100.0,MILLILITRE,345.0,8.0,2026-05-31
5586,May,Pune,d34d554d-671a-42a7-bd48-31301ccbf729,Just Herbs Pigmented Smudge & Sweat Proof Quic...,Makeup & Beauty,Face Makeup,Sindoor,Justherbs,Marico Limited,3.5,GRAM,225.0,49.0,2026-05-31


In [44]:
chain_forecast_zepto_may['projected_qty'].sum()

1280872.0

In [13]:
chain_forecast_zepto = pd.read_csv('/data/aman_singh/acuuracy_check/Marico_Limited_projection_may_zepto.csv')
chain_forecast_zepto['Month'].unique()

array(['Jul', 'Aug', 'Jun'], dtype=object)

In [15]:
chain_forecast_zepto = pd.read_csv('/data/aman_singh/acuuracy_check/Marico_Limited_projection_may_zepto.csv')
chain_forecast_zepto.columns = chain_forecast_zepto.columns.str.lower()
chain_forecast_zepto
chain_forecast_zepto['month'].unique()

month_map = {
    'Jun': '2026-06-30',
    'Aug': '2026-08-31',
    'Jul': '2026-07-31'
}

# Apply mapping
chain_forecast_zepto['date'] = chain_forecast_zepto['month'].map(month_map)
chain_forecast_zepto['date'] = pd.to_datetime(chain_forecast_zepto['date'])
chain_forecast_zepto_june = chain_forecast_zepto[chain_forecast_zepto['date'].isin(['2026-06-30'])]
chain_forecast_zepto_june

,month,cluster_dry,product_variant_id,product_name,category_name,subcategory_name,l3_category_name,brand_name,manufacturer,packsize,unit_of_measure,unit_mrp,projected_qty,date
4,Jun,Chennai,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,260.0,2278,2026-06-30
6,Jun,Bengaluru,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,260.0,4991,2026-06-30
8,Jun,Pune,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,260.0,1389,2026-06-30
10,Jun,Hyderabad,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,260.0,7467,2026-06-30
16,Jun,Ahmedabad,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,260.0,1069,2026-06-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
531,Jun,Kolkata,bdb2302e-6274-4b99-ad19-3339b6cba494,Saffola Muesli Kesar Crunch With Flavour Pops ...,Breakfast & Sauces,Muesli & Oats,Muesli,Saffola Foods,Marico Limited,185.0,GRAM,149.0,1064,2026-06-30
534,Jun,Kolkata,f0e0ef4b-2a02-4e13-8498-35b32f554274,Saffola Mealmaker Soya Chunks Pouch,"Atta, Rice, Oil & Dals",Dals & Pulses,Soya Chunks,Saffola Foods,Marico Limited,400.0,GRAM,111.0,950,2026-06-30
536,Jun,Mumbai,bdb2302e-6274-4b99-ad19-3339b6cba494,Saffola Muesli Kesar Crunch With Flavour Pops ...,Breakfast & Sauces,Muesli & Oats,Muesli,Saffola Foods,Marico Limited,185.0,GRAM,149.0,171,2026-06-30
541,Jun,Hyderabad,f0e0ef4b-2a02-4e13-8498-35b32f554274,Saffola Mealmaker Soya Chunks Pouch,"Atta, Rice, Oil & Dals",Dals & Pulses,Soya Chunks,Saffola Foods,Marico Limited,400.0,GRAM,111.0,1386,2026-06-30


In [20]:
# chain_forecast_zepto = pd.concat([chain_forecast_zepto_april,chain_forecast_zepto_may,chain_forecast_zepto_june])
chain_forecast_zepto = chain_forecast_zepto_june.copy()
chain_forecast_zepto

,month,cluster_dry,product_variant_id,product_name,category_name,subcategory_name,l3_category_name,brand_name,manufacturer,packsize,unit_of_measure,unit_mrp,projected_qty,date
4,Jun,Chennai,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,260.0,2278,2026-06-30
6,Jun,Bengaluru,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,260.0,4991,2026-06-30
8,Jun,Pune,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,260.0,1389,2026-06-30
10,Jun,Hyderabad,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,260.0,7467,2026-06-30
16,Jun,Ahmedabad,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,260.0,1069,2026-06-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
531,Jun,Kolkata,bdb2302e-6274-4b99-ad19-3339b6cba494,Saffola Muesli Kesar Crunch With Flavour Pops ...,Breakfast & Sauces,Muesli & Oats,Muesli,Saffola Foods,Marico Limited,185.0,GRAM,149.0,1064,2026-06-30
534,Jun,Kolkata,f0e0ef4b-2a02-4e13-8498-35b32f554274,Saffola Mealmaker Soya Chunks Pouch,"Atta, Rice, Oil & Dals",Dals & Pulses,Soya Chunks,Saffola Foods,Marico Limited,400.0,GRAM,111.0,950,2026-06-30
536,Jun,Mumbai,bdb2302e-6274-4b99-ad19-3339b6cba494,Saffola Muesli Kesar Crunch With Flavour Pops ...,Breakfast & Sauces,Muesli & Oats,Muesli,Saffola Foods,Marico Limited,185.0,GRAM,149.0,171,2026-06-30
541,Jun,Hyderabad,f0e0ef4b-2a02-4e13-8498-35b32f554274,Saffola Mealmaker Soya Chunks Pouch,"Atta, Rice, Oil & Dals",Dals & Pulses,Soya Chunks,Saffola Foods,Marico Limited,400.0,GRAM,111.0,1386,2026-06-30


In [21]:
chain_forecast_zepto['chain_name'] = 'Zepto'
chain_forecast_zepto.rename(columns = {'product_variant_id':'item_code', 
                                       'projected_qty':'forecast_quantity',
                                       'cluster_dry':'city'}, inplace = True)
chain_forecast_zepto

,month,city,item_code,product_name,category_name,subcategory_name,l3_category_name,brand_name,manufacturer,packsize,unit_of_measure,unit_mrp,forecast_quantity,date,chain_name
4,Jun,Chennai,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,260.0,2278,2026-06-30,Zepto
6,Jun,Bengaluru,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,260.0,4991,2026-06-30,Zepto
8,Jun,Pune,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,260.0,1389,2026-06-30,Zepto
10,Jun,Hyderabad,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,260.0,7467,2026-06-30,Zepto
16,Jun,Ahmedabad,29b306e4-5656-4326-ad7f-4ecd98abf3d8,Parachute Pure Coconut Oil,Hair Care,Hair Oil,Hair Oil,Parachute,Marico Limited,600.0,MILLILITRE,260.0,1069,2026-06-30,Zepto
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
531,Jun,Kolkata,bdb2302e-6274-4b99-ad19-3339b6cba494,Saffola Muesli Kesar Crunch With Flavour Pops ...,Breakfast & Sauces,Muesli & Oats,Muesli,Saffola Foods,Marico Limited,185.0,GRAM,149.0,1064,2026-06-30,Zepto
534,Jun,Kolkata,f0e0ef4b-2a02-4e13-8498-35b32f554274,Saffola Mealmaker Soya Chunks Pouch,"Atta, Rice, Oil & Dals",Dals & Pulses,Soya Chunks,Saffola Foods,Marico Limited,400.0,GRAM,111.0,950,2026-06-30,Zepto
536,Jun,Mumbai,bdb2302e-6274-4b99-ad19-3339b6cba494,Saffola Muesli Kesar Crunch With Flavour Pops ...,Breakfast & Sauces,Muesli & Oats,Muesli,Saffola Foods,Marico Limited,185.0,GRAM,149.0,171,2026-06-30,Zepto
541,Jun,Hyderabad,f0e0ef4b-2a02-4e13-8498-35b32f554274,Saffola Mealmaker Soya Chunks Pouch,"Atta, Rice, Oil & Dals",Dals & Pulses,Soya Chunks,Saffola Foods,Marico Limited,400.0,GRAM,111.0,1386,2026-06-30,Zepto


In [22]:
chain_forecast_zepto.isnull().sum()

month                0
city                 0
item_code            0
product_name         0
category_name        0
subcategory_name     0
l3_category_name     0
brand_name           0
manufacturer         0
packsize             0
unit_of_measure      0
unit_mrp             0
forecast_quantity    0
date                 0
chain_name           0
dtype: int64

In [23]:
chain_forecast_zepto = chain_forecast_zepto.groupby(['chain_name','city','item_code','date'])['forecast_quantity'].sum().reset_index()
chain_forecast_zepto

,chain_name,city,item_code,date,forecast_quantity
0,Zepto,Ahmedabad,292f1828-6db0-4af6-9abd-b8e9095452ae,2026-06-30,1143
1,Zepto,Ahmedabad,29b306e4-5656-4326-ad7f-4ecd98abf3d8,2026-06-30,1069
2,Zepto,Ahmedabad,4d9913da-86d4-4520-9985-1948e50ac1c5,2026-06-30,725
3,Zepto,Ahmedabad,589bce9e-e0bc-467d-a6d0-10838a165e43,2026-06-30,1283
4,Zepto,Ahmedabad,709bd327-baf4-4104-a46d-fdba2b80b99c,2026-06-30,3656
...,...,...,...,...,...
177,Zepto,SAS Nagar,c41c8a9f-f8d7-4aa0-b5bb-39a8905d50bf,2026-06-30,1850
178,Zepto,SAS Nagar,e011578a-374b-406a-890d-f092005c203d,2026-06-30,6309
179,Zepto,SAS Nagar,e2eeab46-1109-41ae-b90b-a9c3a429e62e,2026-06-30,837
180,Zepto,SAS Nagar,e6ab8174-00bd-4a14-b59f-494502928958,2026-06-30,2837


In [24]:
zepto_mapping = pd.read_excel('/data/aman_singh/mt_forecast/Q-com Depot-FC-City Mapping v2.0.xlsx', sheet_name = 'Zepto')
zepto_mapping = zepto_mapping[zepto_mapping['Status'] == 'Active']
zepto_mapping = zepto_mapping[['Channel','City', 'Marico Depot']].drop_duplicates()
zepto_mapping.rename(columns = {'Channel':'platform_name', 'City':'city', 'Marico Depot':'depot'}, inplace = True)
zepto_mapping['platform_name'] = zepto_mapping['platform_name'].str.lower()
zepto_mapping['city'] = zepto_mapping['city'].str.lower()
zepto_mapping

,platform_name,city,depot
0,zepto,ahmedabad,D354
2,zepto,indore,D354
3,zepto,mehsana,D354
4,zepto,rajkot,D354
5,zepto,surat,D354
...,...,...,...
110,zepto,pune,D461
111,zepto,bahadurgarh,D115
112,zepto,gurgaon,NaN
113,zepto,raipur,D465


In [45]:
zepto_mapping[zepto_mapping['city'] == 'NCR']

,platform_name,city,depot


In [86]:
xx = chain_forecast_zepto.copy()

In [25]:
chain_forecast_zepto['city'] = chain_forecast_zepto['city'].str.lower()
chain_forecast_zepto = chain_forecast_zepto.merge(zepto_mapping, on = ['city'], how = 'left')
chain_forecast_zepto

,chain_name,city,item_code,date,forecast_quantity,platform_name,depot
0,Zepto,ahmedabad,292f1828-6db0-4af6-9abd-b8e9095452ae,2026-06-30,1143,zepto,D354
1,Zepto,ahmedabad,29b306e4-5656-4326-ad7f-4ecd98abf3d8,2026-06-30,1069,zepto,D354
2,Zepto,ahmedabad,4d9913da-86d4-4520-9985-1948e50ac1c5,2026-06-30,725,zepto,D354
3,Zepto,ahmedabad,589bce9e-e0bc-467d-a6d0-10838a165e43,2026-06-30,1283,zepto,D354
4,Zepto,ahmedabad,709bd327-baf4-4104-a46d-fdba2b80b99c,2026-06-30,3656,zepto,D354
...,...,...,...,...,...,...,...
177,Zepto,sas nagar,c41c8a9f-f8d7-4aa0-b5bb-39a8905d50bf,2026-06-30,1850,zepto,D115
178,Zepto,sas nagar,e011578a-374b-406a-890d-f092005c203d,2026-06-30,6309,zepto,D115
179,Zepto,sas nagar,e2eeab46-1109-41ae-b90b-a9c3a429e62e,2026-06-30,837,zepto,D115
180,Zepto,sas nagar,e6ab8174-00bd-4a14-b59f-494502928958,2026-06-30,2837,zepto,D115


In [26]:
chain_forecast_zepto[chain_forecast_zepto['depot'].isna()]#['city'].unique()

,chain_name,city,item_code,date,forecast_quantity,platform_name,depot


In [29]:
len_before_merge = len(chain_forecast_zepto)
chain_forecast_zepto['item_code'] = chain_forecast_zepto['item_code'].astype(str)
mapping['asin'] = mapping['asin'].astype(str)
temp = mapping[['platform_name','asin','EAN','PSKU','UOM','Vol per unit']].drop_duplicates()

temp = temp[temp['platform_name'].isin(['Blinkit', 'Swiggy', 'Zepto'])]
#temp['platform_name'].unique()
duplicates = temp[temp.duplicated(subset="asin", keep=False)]
duplicates



,platform_name,asin,EAN,PSKU,UOM,Vol per unit


In [30]:
temp['PSKU'] = temp['PSKU'].astype(str)
temp['EAN'] = temp['EAN'].astype(str)
temp['UOM'] = temp['UOM'].astype(str)
len_before_merge = len(chain_forecast_zepto)
chain_forecast_zepto = chain_forecast_zepto.merge(temp,
                  left_on = ['item_code'], right_on = ['asin'], how = 'left')
assert(len_before_merge == len(chain_forecast_zepto))
chain_forecast_zepto['date'] = pd.to_datetime(chain_forecast_zepto['date'])
chain_forecast_zepto

,chain_name,city,item_code,date,forecast_quantity,platform_name_x,depot,platform_name_y,asin,EAN,PSKU,UOM,Vol per unit
0,Zepto,ahmedabad,292f1828-6db0-4af6-9abd-b8e9095452ae,2026-06-30,1143,zepto,D354,Zepto,292f1828-6db0-4af6-9abd-b8e9095452ae,8901088155496,718464,TO,500.0
1,Zepto,ahmedabad,29b306e4-5656-4326-ad7f-4ecd98abf3d8,2026-06-30,1069,zepto,D354,Zepto,29b306e4-5656-4326-ad7f-4ecd98abf3d8,8901088102872,718825,KL,600.0
2,Zepto,ahmedabad,4d9913da-86d4-4520-9985-1948e50ac1c5,2026-06-30,725,zepto,D354,Zepto,4d9913da-86d4-4520-9985-1948e50ac1c5,8901088064590,718553,L,400.0
3,Zepto,ahmedabad,589bce9e-e0bc-467d-a6d0-10838a165e43,2026-06-30,1283,zepto,D354,Zepto,589bce9e-e0bc-467d-a6d0-10838a165e43,8901088171755,719162,TO,250.0
4,Zepto,ahmedabad,709bd327-baf4-4104-a46d-fdba2b80b99c,2026-06-30,3656,zepto,D354,Zepto,709bd327-baf4-4104-a46d-fdba2b80b99c,8901088136945,718828,KL,300.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
177,Zepto,sas nagar,c41c8a9f-f8d7-4aa0-b5bb-39a8905d50bf,2026-06-30,1850,zepto,D115,Zepto,c41c8a9f-f8d7-4aa0-b5bb-39a8905d50bf,8901088774529,727811,L,9.0
178,Zepto,sas nagar,e011578a-374b-406a-890d-f092005c203d,2026-06-30,6309,zepto,D115,Zepto,e011578a-374b-406a-890d-f092005c203d,8901088068734,718559,TO,38.0
179,Zepto,sas nagar,e2eeab46-1109-41ae-b90b-a9c3a429e62e,2026-06-30,837,zepto,D115,Zepto,e2eeab46-1109-41ae-b90b-a9c3a429e62e,8901088017381,718341,KL,1000.0
180,Zepto,sas nagar,e6ab8174-00bd-4a14-b59f-494502928958,2026-06-30,2837,zepto,D115,Zepto,e6ab8174-00bd-4a14-b59f-494502928958,8901088068758,718560,TO,38.0


In [31]:
xx = chain_forecast_zepto.copy()

In [32]:
chain_forecast_zepto.isnull().sum()

chain_name           0
city                 0
item_code            0
date                 0
forecast_quantity    0
platform_name_x      0
depot                0
platform_name_y      0
asin                 0
EAN                  0
PSKU                 0
UOM                  0
Vol per unit         0
dtype: int64

In [33]:
chain_forecast_zepto.dropna(subset = ['PSKU'],inplace = True)
chain_forecast_zepto

,chain_name,city,item_code,date,forecast_quantity,platform_name_x,depot,platform_name_y,asin,EAN,PSKU,UOM,Vol per unit
0,Zepto,ahmedabad,292f1828-6db0-4af6-9abd-b8e9095452ae,2026-06-30,1143,zepto,D354,Zepto,292f1828-6db0-4af6-9abd-b8e9095452ae,8901088155496,718464,TO,500.0
1,Zepto,ahmedabad,29b306e4-5656-4326-ad7f-4ecd98abf3d8,2026-06-30,1069,zepto,D354,Zepto,29b306e4-5656-4326-ad7f-4ecd98abf3d8,8901088102872,718825,KL,600.0
2,Zepto,ahmedabad,4d9913da-86d4-4520-9985-1948e50ac1c5,2026-06-30,725,zepto,D354,Zepto,4d9913da-86d4-4520-9985-1948e50ac1c5,8901088064590,718553,L,400.0
3,Zepto,ahmedabad,589bce9e-e0bc-467d-a6d0-10838a165e43,2026-06-30,1283,zepto,D354,Zepto,589bce9e-e0bc-467d-a6d0-10838a165e43,8901088171755,719162,TO,250.0
4,Zepto,ahmedabad,709bd327-baf4-4104-a46d-fdba2b80b99c,2026-06-30,3656,zepto,D354,Zepto,709bd327-baf4-4104-a46d-fdba2b80b99c,8901088136945,718828,KL,300.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
177,Zepto,sas nagar,c41c8a9f-f8d7-4aa0-b5bb-39a8905d50bf,2026-06-30,1850,zepto,D115,Zepto,c41c8a9f-f8d7-4aa0-b5bb-39a8905d50bf,8901088774529,727811,L,9.0
178,Zepto,sas nagar,e011578a-374b-406a-890d-f092005c203d,2026-06-30,6309,zepto,D115,Zepto,e011578a-374b-406a-890d-f092005c203d,8901088068734,718559,TO,38.0
179,Zepto,sas nagar,e2eeab46-1109-41ae-b90b-a9c3a429e62e,2026-06-30,837,zepto,D115,Zepto,e2eeab46-1109-41ae-b90b-a9c3a429e62e,8901088017381,718341,KL,1000.0
180,Zepto,sas nagar,e6ab8174-00bd-4a14-b59f-494502928958,2026-06-30,2837,zepto,D115,Zepto,e6ab8174-00bd-4a14-b59f-494502928958,8901088068758,718560,TO,38.0


In [ ]:
# chain_forecast_zepto.duplicated(subset=['chain_name','PSKU','date'], keep=False).sum()

2960

In [ ]:
chain_forecast_zepto['month_date'] = chain_forecast_zepto['date'] + pd.offsets.MonthEnd(0)

chain_forecast_zepto.rename(columns = {'item_code':'platform_code', 'EAN':'eancode', 'UOM':'uom_reporting',
                         'Vol per unit':'vol_per_unit'},inplace=True)
chain_forecast_zepto['vol_in_lit'] = chain_forecast_zepto['forecast_quantity']*chain_forecast_zepto['vol_per_unit']/1000
chain_forecast_zepto['vol_in_rum'] = chain_forecast_zepto.apply(
    lambda x: x['vol_in_lit'] / 1000 if x['uom_reporting'] in ['KL', 'TO'] else x['vol_in_lit'],
    axis=1
)

chain_forecast_zepto = chain_forecast_zepto.groupby(['chain_name','depot','PSKU','month_date'])[['vol_in_rum','forecast_quantity']].sum().reset_index()
chain_forecast_zepto['PSKU'] = chain_forecast_zepto['PSKU'].astype(int)
chain_forecast_zepto.rename(columns = {'PSKU':'parent_material_code'}, inplace = True)
chain_forecast_zepto

,chain_name,depot,parent_material_code,month_date,vol_in_rum,forecast_quantity
0,Zepto,D112,718287,2026-06-30,1.087400,5437
1,Zepto,D112,718341,2026-06-30,10.389000,10389
2,Zepto,D112,718371,2026-06-30,319.350000,6387
3,Zepto,D112,718398,2026-06-30,12.946412,13891
4,Zepto,D112,718464,2026-06-30,2.040500,4081
...,...,...,...,...,...,...
177,Zepto,D674,718560,2026-06-30,0.131100,3450
178,Zepto,D674,718825,2026-06-30,1.366800,2278
179,Zepto,D674,718828,2026-06-30,3.609600,12032
180,Zepto,D674,721427,2026-06-30,1.673000,1673


In [39]:
chain_forecast_zepto['forecast_quantity'].sum()

753604

In [77]:
chain_forecast_zepto.to_csv('forecast_zepto.csv')

In [52]:
df_chk

,chain_name,FC,parent_material_code,month_date,vol_in_rum,depot_code
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-08-31,6.2040,D354
1,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-08-31,0.5550,D354
2,Blinkit,ahmedabad a2 - feeder warehouse,718322,2026-08-31,1.4800,D354
3,Blinkit,ahmedabad a2 - feeder warehouse,718328,2026-08-31,0.0522,D354
4,Blinkit,ahmedabad a2 - feeder warehouse,718341,2026-08-31,1.6820,D354
...,...,...,...,...,...,...
11020,Swiggy,viz im1,810522,2026-08-31,0.0240,D572
11021,Swiggy,viz im1,810673,2026-08-31,0.5040,D572
11022,Swiggy,viz im1,810674,2026-08-31,0.5040,D572
11023,Swiggy,viz im1,810685,2026-08-31,0.0080,D572


In [53]:
fc_depot_mapping = {
    'indore i2 - feeder warehouse': 'D464',
    'mumbai m12 - feeder warehouse': 'D356',
    'patna p2 - feeder warehouse': 'D233',
    'pune p3 - feeder warehouse': 'D461',
    'ranchi r2 - feeder warehouse': 'D234',
    'blr im4': 'D673'
}

# Fill only null depot codes
df_chk.loc[df_chk['depot_code'].isna(), 'depot_code'] = (
    df_chk.loc[df_chk['depot_code'].isna(), 'FC']
          .str.lower()
          .map(fc_depot_mapping)
)
df_chk

,chain_name,FC,parent_material_code,month_date,vol_in_rum,depot_code
0,Blinkit,ahmedabad a2 - feeder warehouse,718288,2026-08-31,6.2040,D354
1,Blinkit,ahmedabad a2 - feeder warehouse,718312,2026-08-31,0.5550,D354
2,Blinkit,ahmedabad a2 - feeder warehouse,718322,2026-08-31,1.4800,D354
3,Blinkit,ahmedabad a2 - feeder warehouse,718328,2026-08-31,0.0522,D354
4,Blinkit,ahmedabad a2 - feeder warehouse,718341,2026-08-31,1.6820,D354
...,...,...,...,...,...,...
11020,Swiggy,viz im1,810522,2026-08-31,0.0240,D572
11021,Swiggy,viz im1,810673,2026-08-31,0.5040,D572
11022,Swiggy,viz im1,810674,2026-08-31,0.5040,D572
11023,Swiggy,viz im1,810685,2026-08-31,0.0080,D572


In [55]:
df_chk = df_chk.groupby(['chain_name','depot_code','parent_material_code','month_date'])['vol_in_rum'].sum().reset_index()
df_chk

,chain_name,depot_code,parent_material_code,month_date,vol_in_rum
0,Blinkit,D112,718288,2026-08-31,0.0420
1,Blinkit,D112,718312,2026-08-31,1.9460
2,Blinkit,D112,718322,2026-08-31,3.5400
3,Blinkit,D112,718328,2026-08-31,0.2151
4,Blinkit,D112,718330,2026-08-31,0.0250
...,...,...,...,...,...
6539,Swiggy,D677,810521,2026-08-31,0.0240
6540,Swiggy,D677,810522,2026-08-31,0.0240
6541,Swiggy,D677,810673,2026-08-31,0.0000
6542,Swiggy,D677,810674,2026-08-31,0.0000


In [56]:
df_chk.rename(columns = {'depot_code':'depot'},inplace = True)
final_df = pd.concat([df_chk,chain_forecast_zepto])
final_df

,chain_name,depot,parent_material_code,month_date,vol_in_rum
0,Blinkit,D112,718288,2026-08-31,0.0420
1,Blinkit,D112,718312,2026-08-31,1.9460
2,Blinkit,D112,718322,2026-08-31,3.5400
3,Blinkit,D112,718328,2026-08-31,0.2151
4,Blinkit,D112,718330,2026-08-31,0.0250
...,...,...,...,...,...
1245,Zepto,D674,810518,2026-08-31,0.3080
1246,Zepto,D674,810519,2026-08-31,0.1000
1247,Zepto,D674,810673,2026-08-31,0.5040
1248,Zepto,D674,810674,2026-08-31,0.3360


In [ ]:
final_df.to_csv('qcom_chain_forecast_aug.csv')

### The end

In [71]:
primary = pd.read_csv('/data/aman_singh/acuuracy_check/QCOM Chain PSKU OTP Output/live_runs/QCOM Chain Depot PSKU Primary_live_runs_06_Jul_2026.csv')
primary

,Key2,Key,Chain,Depot,PSKU,Brand,Index Rate,Portfolio,Run Month,Month Date,...,Offtake Chain depot PSKU Lag 3 Val,Offtake Chain depot PSKU P3M Val,LY Offtake Chain depot PSKU P3M Val,LY Offtake Chain depot PSKU Actuals Val,LY Offtake Chain depot PSKU Lag 1 Val,LY Offtake Chain depot PSKU Lag 2 Val,LY Offtake Chain depot PSKU Lag 3 Val,LY Offtake Chain depot PSKU Lead 1 Val,LY Offtake Chain depot PSKU Lead 2 Val,Calculated Primary Val
0,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-06-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-07-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-08-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-09-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-10-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212675,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2026-11-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
212676,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2026-12-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
212677,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2027-01-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
212678,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2027-02-28,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [72]:
primary.columns

Index(['Key2', 'Key', 'Chain', 'Depot', 'PSKU', 'Brand', 'Index Rate',
       'Portfolio', 'Run Month', 'Month Date',
       ...
       'Offtake Chain depot PSKU Lag 3 Val',
       'Offtake Chain depot PSKU P3M Val',
       'LY Offtake Chain depot PSKU P3M Val',
       'LY Offtake Chain depot PSKU Actuals Val',
       'LY Offtake Chain depot PSKU Lag 1 Val',
       'LY Offtake Chain depot PSKU Lag 2 Val',
       'LY Offtake Chain depot PSKU Lag 3 Val',
       'LY Offtake Chain depot PSKU Lead 1 Val',
       'LY Offtake Chain depot PSKU Lead 2 Val', 'Calculated Primary Val'],
      dtype='object', length=117)

In [73]:
final_df.columns = ['Chain', 'Depot', 'PSKU','Month Date','Chain_primary_vol']
final_df['Depot'] = final_df['Depot'].str.lower()
final_df

,Chain,Depot,PSKU,Month Date,Chain_primary_vol
0,Blinkit,d112,718288,2026-08-31,0.0420
1,Blinkit,d112,718312,2026-08-31,1.9460
2,Blinkit,d112,718322,2026-08-31,3.5400
3,Blinkit,d112,718328,2026-08-31,0.2151
4,Blinkit,d112,718330,2026-08-31,0.0250
...,...,...,...,...,...
1245,Zepto,d674,810518,2026-08-31,0.3080
1246,Zepto,d674,810519,2026-08-31,0.1000
1247,Zepto,d674,810673,2026-08-31,0.5040
1248,Zepto,d674,810674,2026-08-31,0.3360


In [74]:
primary['Month Date'] = pd.to_datetime(primary['Month Date'])
primary = primary.merge(final_df, on = ['Chain', 'Depot', 'PSKU','Month Date'], how = 'left')
primary['Chain_primary_val'] = primary['Chain_primary_vol']*primary['Index Rate']/10**7
primary

,Key2,Key,Chain,Depot,PSKU,Brand,Index Rate,Portfolio,Run Month,Month Date,...,LY Offtake Chain depot PSKU P3M Val,LY Offtake Chain depot PSKU Actuals Val,LY Offtake Chain depot PSKU Lag 1 Val,LY Offtake Chain depot PSKU Lag 2 Val,LY Offtake Chain depot PSKU Lag 3 Val,LY Offtake Chain depot PSKU Lead 1 Val,LY Offtake Chain depot PSKU Lead 2 Val,Calculated Primary Val,Chain_primary_vol,Chain_primary_val
0,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-06-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
1,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-07-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
2,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-08-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
3,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-09-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
4,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-10-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212675,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2026-11-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
212676,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2026-12-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
212677,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2027-01-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
212678,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2027-02-28,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN


In [75]:
primary[primary['Month Date']=='2026-08-31']['Chain_primary_val'].sum()

41.958339936520986

In [76]:
primary.to_csv('cdp_c_forecast.csv')

In [66]:
x['Chain_primary_vol'].isnull().sum()

207123

In [67]:
primary

,Key2,Key,Chain,Depot,PSKU,Brand,Index Rate,Portfolio,Run Month,Month Date,...,Offtake Chain depot PSKU Lag 3 Val,Offtake Chain depot PSKU P3M Val,LY Offtake Chain depot PSKU P3M Val,LY Offtake Chain depot PSKU Actuals Val,LY Offtake Chain depot PSKU Lag 1 Val,LY Offtake Chain depot PSKU Lag 2 Val,LY Offtake Chain depot PSKU Lag 3 Val,LY Offtake Chain depot PSKU Lead 1 Val,LY Offtake Chain depot PSKU Lead 2 Val,Calculated Primary Val
0,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-06-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-07-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-08-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-09-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,blinkit_d112_718287,blinkit_d112_718287,Blinkit,d112,718287,PCNO(R),349274.00142,CNO,2026-07-31,2026-10-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212675,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2026-11-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
212676,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2026-12-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
212677,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2027-01-31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
212678,zepto_nan_811287,zepto_nan_811287,Zepto,NaN,811287,PA_RSW_SR,740.00000,NaN,2026-07-31,2027-02-28,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# swiggy_unpivoted.groupby(['chain_name','facility_name','item_code','date'])['forecast_quantity'].sum().reset_index()

,chain_name,facility_name,item_code,date,forecast_quantity
0,Swiggy,AHM DELHIVERY,3,2026-07-31,192
1,Swiggy,AHM DELHIVERY,3,2026-08-31,768
2,Swiggy,AHM DELHIVERY,3,2026-09-30,576
3,Swiggy,AHM DELHIVERY,102,2026-07-31,60
4,Swiggy,AHM DELHIVERY,102,2026-08-31,160
...,...,...,...,...,...
29878,Swiggy,VIZ IM1,995855,2026-08-31,0
29879,Swiggy,VIZ IM1,995855,2026-09-30,0
29880,Swiggy,VIZ IM1,999977,2026-07-31,4
29881,Swiggy,VIZ IM1,999977,2026-08-31,5


In [ ]:
blinkit_unpivoted = blinkit_unpivoted[['chain_name','facility_name','item_code','date','forecast_quantity']]
swiggy_unpivoted = swiggy_unpivoted[['chain_name','facility_name','item_code','date','forecast_quantity']]


In [ ]:
chain_forecast_unpivoted = pd.concat([blinkit_unpivoted,swiggy_unpivoted])
chain_forecast_unpivoted

,chain_name,facility_name,item_code,date,forecast_quantity
0,Blinkit,Surat S1 - Feeder Warehouse,10171388,2026-07-31,856
1,Blinkit,Surat S1 - Feeder Warehouse,10232351,2026-07-31,18
2,Blinkit,Surat S1 - Feeder Warehouse,10029462,2026-07-31,12
3,Blinkit,Surat S1 - Feeder Warehouse,10015827,2026-07-31,209
4,Blinkit,Surat S1 - Feeder Warehouse,10116052,2026-07-31,65
...,...,...,...,...,...
29878,Swiggy,PUN DELHIVERY,990631,2026-09-30,22
29879,Swiggy,CHD ECOM,991861,2026-09-30,24
29880,Swiggy,CHN ECOM,995855,2026-09-30,0
29881,Swiggy,HYD IM1,998784,2026-09-30,0


In [ ]:
len_before_merge = len(chain_forecast_unpivoted)
chain_forecast_unpivoted['item_code'] = chain_forecast_unpivoted['item_code'].astype(str)
mapping['asin'] = mapping['asin'].astype(str)
temp = mapping[['platform_name','asin','EAN','PSKU','UOM','Vol per unit']].drop_duplicates()

temp = temp[temp['platform_name'].isin(['Blinkit', 'Swiggy', 'Zepto'])]
#temp['platform_name'].unique()
duplicates = temp[temp.duplicated(subset="asin", keep=False)]
duplicates



,platform_name,asin,EAN,PSKU,UOM,Vol per unit


In [ ]:
# # Keys to match rows on
# keys = ["platform_name", "asin", "EAN", "PSKU", "UOM", "Vol per unit"]

# # Build a small DataFrame with the rows to drop
# rows_to_drop = pd.DataFrame([
#     {
#         "platform_name": "Zepto",
#         "asin": "0523a4ba-32cf-4e59-abd8-0e4086859b39",
#         "EAN": "8901088205924",
#         "PSKU": "718729",
#         "UOM": "L",
#         "Vol per unit": 100.0,
#     },
#     {
#         "platform_name": "Zepto",
#         "asin": "197827dc-3184-4c57-a966-5461967bcb3a",
#         "EAN": "8901088884402",
#         "PSKU": "808485",
#         "UOM": "L",
#         "Vol per unit": 150.0,
#     },
#     {
#         "platform_name": "Zepto",
#         "asin": "82d8e93d-3d18-44b3-9904-3bcf521d0204",
#         "EAN": "8906051370753",
#         "PSKU": "807069",
#         "UOM": "L",
#         "Vol per unit": 150.0,
#     },
#     {
#         "platform_name": "Zepto",
#         "asin": "82d8e93d-3d18-44b3-9904-3bcf521d0204",
#         "EAN": "8901088075817",
#         "PSKU": "808262",
#         "UOM": "L",
#         "Vol per unit": 150.0,
#     },
#     {
#         "platform_name": "Swiggy",
#         "asin": "944906",
#         "EAN": "8901088150095",
#         "PSKU": "718976",
#         "UOM": "L",
#         "Vol per unit": 300.0,
#     },
# ])

# # Mark rows to drop via left-merge on keys
# _marked = temp.merge(
#     rows_to_drop.assign(_drop=1),
#     on=keys,
#     how="left",
#     validate="m:m"  # remove if unsure about duplicates
# )

# # Keep everything that was not marked to drop
# temp_clean = _marked[_marked["_drop"].isna()].drop(columns=["_drop"])
# temp_clean
# duplicates = temp_clean[temp_clean.duplicated(subset="asin", keep=False)]
# duplicates

In [ ]:
temp['PSKU'] = temp['PSKU'].astype(str)
temp['EAN'] = temp['EAN'].astype(str)
temp['UOM'] = temp['UOM'].astype(str)
len_before_merge = len(chain_forecast_unpivoted)
df_chk = chain_forecast_unpivoted.merge(temp,
                  left_on = ['item_code'], right_on = ['asin'], how = 'left')
assert(len_before_merge == len(df_chk))
df_chk['date'] = pd.to_datetime(df_chk['date'])

In [ ]:
df_chk

,chain_name,facility_name,item_code,date,forecast_quantity,platform_name,asin,EAN,PSKU,UOM,Vol per unit
0,Blinkit,Surat S1 - Feeder Warehouse,10171388,2026-07-31,856,Blinkit,10171388,8901088213608,721427,TO,1000.0
1,Blinkit,Surat S1 - Feeder Warehouse,10232351,2026-07-31,18,Blinkit,10232351,8901088796804,810520,KL,1000.0
2,Blinkit,Surat S1 - Feeder Warehouse,10029462,2026-07-31,12,Blinkit,10029462,6001159111856,807033,L,125.0
3,Blinkit,Surat S1 - Feeder Warehouse,10015827,2026-07-31,209,Blinkit,10015827,8901088043953,718312,KL,1000.0
4,Blinkit,Surat S1 - Feeder Warehouse,10116052,2026-07-31,65,Blinkit,10116052,8901088205993,721133,L,300.0
...,...,...,...,...,...,...,...,...,...,...,...
50086,Swiggy,PUN DELHIVERY,990631,2026-09-30,22,NaN,NaN,NaN,NaN,NaN,NaN
50087,Swiggy,CHD ECOM,991861,2026-09-30,24,Swiggy,991861,8906027074531,729893,L,240.0
50088,Swiggy,CHN ECOM,995855,2026-09-30,0,NaN,NaN,NaN,NaN,NaN,NaN
50089,Swiggy,HYD IM1,998784,2026-09-30,0,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
duplicates = df_chk[df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False)]
duplicates.isnull().sum()

chain_name               0
facility_name            0
item_code                0
date                     0
forecast_quantity        0
platform_name        12111
asin                 12111
EAN                  12111
PSKU                 12111
UOM                  12111
Vol per unit         12111
dtype: int64

In [ ]:
df_chk = df_chk.dropna(subset = ['PSKU'])
df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False).sum()

474

In [ ]:
# duplicates = df_chk[df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False)]
# duplicates.sort_values(by=['chain_name','facility_name','PSKU','date'])[:60]

,chain_name,facility_name,item_code,date,forecast_quantity,platform_name,asin,EAN,PSKU,UOM,Vol per unit
20560,Swiggy,AHM DELHIVERY,554793,2026-07-31,0,Swiggy,554793,8901088886970,809042,KG,225.0
23718,Swiggy,AHM DELHIVERY,60103,2026-07-31,60,Swiggy,60103,8901088886970,809042,KG,225.0
30521,Swiggy,AHM DELHIVERY,554793,2026-08-31,0,Swiggy,554793,8901088886970,809042,KG,225.0
33679,Swiggy,AHM DELHIVERY,60103,2026-08-31,60,Swiggy,60103,8901088886970,809042,KG,225.0
40482,Swiggy,AHM DELHIVERY,554793,2026-09-30,0,Swiggy,554793,8901088886970,809042,KG,225.0
43640,Swiggy,AHM DELHIVERY,60103,2026-09-30,120,Swiggy,60103,8901088886970,809042,KG,225.0
21827,Swiggy,BLR DHL,819548,2026-07-31,192,Swiggy,819548,8901088171755,719162,TO,250.0
23859,Swiggy,BLR DHL,298412,2026-07-31,0,Swiggy,298412,8901088171755,719162,TO,250.0
31788,Swiggy,BLR DHL,819548,2026-08-31,192,Swiggy,819548,8901088171755,719162,TO,250.0
33820,Swiggy,BLR DHL,298412,2026-08-31,0,Swiggy,298412,8901088171755,719162,TO,250.0


In [ ]:
# duplicates.sort_values(by=['chain_name','facility_name','PSKU','date']).to_csv('duplicates_swiggy2.csv')

In [ ]:
# df_chk = df_chk.sort_values('forecast_quantity', ascending=False) \
#        .drop_duplicates(subset=['chain_name', 'facility_name','PSKU' , 'date'], keep='first')
# df_chk.duplicated(subset=['chain_name','facility_name','PSKU','date'], keep=False).sum()

0

In [ ]:
df_chk.columns

Index(['chain_name', 'facility_name', 'item_code', 'date', 'forecast_quantity',
       'platform_name', 'asin', 'EAN', 'PSKU', 'UOM', 'Vol per unit'],
      dtype='object')

In [ ]:
df_chk['month_date'] = df_chk['date'] + pd.offsets.MonthEnd(0)

df_chk.rename(columns = {'item_code':'platform_code', 'EAN':'eancode', 'UOM':'uom_reporting',
                         'Vol per unit':'vol_per_unit'},inplace=True)
df_chk['vol_in_lit'] = df_chk['forecast_quantity']*df_chk['vol_per_unit']/1000
df_chk['vol_in_rum'] = df_chk.apply(
    lambda x: x['vol_in_lit'] / 1000 if x['uom_reporting'] in ['KL', 'TO'] else x['vol_in_lit'],
    axis=1
)

df_chk = df_chk.groupby(['chain_name','facility_name', 'PSKU','month_date'])[['vol_in_rum']].sum().reset_index()
df_chk['PSKU'] = df_chk['PSKU'].astype(int)
df_chk.rename(columns = {'PSKU':'parent_material_code'}, inplace = True)
df_chk

,chain_name,facility_name,parent_material_code,month_date,vol_in_rum,forecast_quantity
0,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-07-31,5.754,959
1,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-08-31,6.204,1034
2,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-09-30,6.222,1037
3,Blinkit,Ahmedabad A2 - Feeder Warehouse,718288,2026-10-31,8.568,1428
4,Blinkit,Ahmedabad A2 - Feeder Warehouse,718312,2026-07-31,0.501,501
...,...,...,...,...,...,...
37674,Swiggy,VIZ IM1,810685,2026-08-31,0.008,20
37675,Swiggy,VIZ IM1,810685,2026-09-30,0.008,20
37676,Swiggy,VIZ IM1,810738,2026-07-31,0.000,0
37677,Swiggy,VIZ IM1,810738,2026-08-31,0.000,0


In [ ]:
df_chk.duplicated(subset=['chain_name','facility_name','parent_material_code','month_date'], keep=False).sum()

0